# SMOOTH — Python demo

**S**urrogate **M**ap **O**ptimized **O**bservation and **T**opographic **H**ypothesis testing:
group-level statistical inference on human intracranial EEG (ECoG/sEEG).

iEEG electrode coverage is sparse, uneven, and different in every subject. SMOOTH builds null
distributions from **spatially-constrained surrogate maps** — each subject's cortical map is decomposed
into graph-Laplacian eigenmodes, the modal coefficients are randomly sign-flipped, and the map is
reconstructed. This preserves each subject's intrinsic spatial smoothness while destroying topographic
alignment across subjects, so a cluster test asks whether observed spatial convergence exceeds chance
without relying on a predefined atlas.

This notebook mirrors the MATLAB `demo/SMOOTHdemo.m` using the pure-Python port `python/smoothstat.py`
(numpy + scipy only). It reproduces the two significant clusters of the primary social-cognition dataset.

**Requirements:** `numpy`, `scipy`, `matplotlib`, and FreeSurfer surfaces (set `SUBJECTS_DIR` or
`FREESURFER_HOME`). We use `fsaverage5` here for speed; `fsaverage` reproduces the manuscript.

In [ ]:
import os, sys
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt

# make the port importable + point at FreeSurfer surfaces
sys.path.insert(0, os.path.join(os.getcwd(), "..", "python"))
os.environ.setdefault("SUBJECTS_DIR", "/Applications/freesurfer/7.3.2/subjects")  # <-- edit if needed
import smoothstat as ss

## 1. Load the demo dataset
Each subject contributes a per-channel dependent variable (`stat`) and electrode positions
(`elec.chanpos`) in fsaverage coordinates. For the primary dataset the DV is a permutation z-statistic
quantifying each channel's fit with a mentalizing regressor.

In [ ]:
src = loadmat(os.path.join("..", "demo", "source.mat"), squeeze_me=True, struct_as_record=False)["source"]
subs = [(np.asarray(s.elec.chanpos, float), np.asarray(s.stat, float).ravel()) for s in np.atleast_1d(src)]
print(f"{len(subs)} subjects, {sum(len(cp) for cp,_ in subs)} total electrodes")

## 2. Run SMOOTHstat
`run_smoothstat` builds each subject's smoother + eigenbasis, computes the group one-sample t-map over
covered vertices, and calibrates cluster masses against the sign-flip surrogate null. We use 200
permutations here for speed (use `numrand=1000` for reported p-values) and a one-tailed positive test.

In [ ]:
surf, nL, nR, adj = ss.load_mesh("fsaverage5")          # "fsaverage" to match the manuscript
res = ss.run_smoothstat(subs, surf, nL, nR, adj, numrand=200, tail="positive", seed=42, verbose=False)

print(f"coverage: up to {int(res['coverage'].max())} subjects/vertex | "
      f"{int((~res['mask']).sum())} testable vertices\n")
print(f"{'mass':>10}  {'p':>7}  {'n_vtx':>6}")
for mass, p, mem in res["posclusters"][:6]:
    star = ' *' if p <= 0.05 else ''
    print(f"{mass:10.1f}  {p:7.4f}  {len(mem):6d}{star}")

## 3. Visualize the group t-map and significant clusters
Covered vertices coloured by the group t-statistic (RdBu, ±4); vertices in a significant cluster
(p ≤ 0.05) outlined in black. The two clusters fall in left temporoparietal and right TPJ cortex.

In [ ]:
t   = res["stat"]; sig = res["sig"]; cov = ~res["mask"]
pos = np.vstack([surf["lh"]["pial"], surf["rh"]["pial"]])
m   = cov & np.isfinite(t)

fig = plt.figure(figsize=(14, 4.5))
for k, (elev, azim, ttl) in enumerate([(0, 180, "left lateral"), (90, -90, "dorsal"), (0, 0, "right lateral")]):
    ax = fig.add_subplot(1, 3, k+1, projection="3d")
    sc = ax.scatter(pos[m,0], pos[m,1], pos[m,2], c=t[m], cmap="RdBu_r", vmin=-4, vmax=4, s=2, alpha=0.6)
    ax.scatter(pos[sig,0], pos[sig,1], pos[sig,2], c="none", edgecolors="k", s=10, linewidths=0.4)
    ax.view_init(elev=elev, azim=azim); ax.set_axis_off(); ax.set_title(ttl)
    ax.set_box_aspect((np.ptp(pos[:,0]), np.ptp(pos[:,1]), np.ptp(pos[:,2])))
fig.colorbar(sc, ax=fig.axes, shrink=0.5, label="group t"); plt.show()

## 4. Surrogate spatial-autocorrelation check
The surrogate maps must reproduce each subject's spatial smoothness. Per-subject Moran's I of the
empirical vs surrogate maps should fall on the identity line — the null preserves smoothness while
scrambling topography.

In [ ]:
emp, sur_mean, sur_sd = ss.moran_sa(res, adj, nsurr=10)
lim = [float(np.nanmin([emp, sur_mean])*0.98), float(np.nanmax([emp, sur_mean])*1.02)]
plt.figure(figsize=(4.5, 4.5))
plt.plot(lim, lim, "k--", lw=1)
plt.errorbar(sur_mean, emp, xerr=sur_sd, fmt="o", ms=5, color="#2e7d32", capsize=2)
plt.xlabel("surrogate Moran's I"); plt.ylabel("empirical Moran's I")
plt.title("surrogates preserve spatial autocorrelation"); plt.axis("square"); plt.xlim(lim); plt.ylim(lim); plt.show()

## Notes
- **Reproduce the manuscript**: use `ss.load_mesh("fsaverage")` and `numrand=1000`. On `fsaverage5`/200
  perms the two clusters still emerge with the correct rank and significance; exact p-values are coarser.
- Electrode coordinates must be in **fsaverage** space (see `python/README.md`).